In [1]:
import os
import pandas as pd
import numpy as np
import scipy
from scipy.stats import spearmanr

import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib import rc
import seaborn as sns
import Bio.PDB
from scipy import stats
from Bio.PDB import SASA # SASA module won't load from newest version of Biopython for some reason
# import Geometry # Biopython Geometry module still not on main branch

import time
import sys
import glob
sys.path.append('/data/mhoffert/fiererlab/adenylate_kinase_ogt/protein_utils/')
from collections import Counter

import Geometry

from IPython.display import display, clear_output

In [2]:
from __future__ import print_function
import warnings
warnings.filterwarnings('ignore') # make the notebook nicer


# import os
# os.environ['QT_QPA_PLATFORM']='offscreen'

import nglview as nv
import pytraj as pt

print("nglview version = {}".format(nv.__version__))
print("pytraj version = {}".format(pt.__version__))

from Bio.PDB import PDBParser, Select, PDBIO, Polypeptide
from Bio.SeqUtils import seq1

nglview version = 3.1.1
pytraj version = 2.0.6


## Data

In [13]:
# combine rosetta data
rosetta_files = glob.glob('/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/rosetta_out/*_processed_rosetta_out.sc')

In [14]:
rosetta_data = []
for i, r in enumerate(rosetta_files):
    if i % 100 == 0:
        display(i)
        clear_output(wait=True)
    rosetta_data.append(pd.read_csv(r, delim_whitespace=True, skiprows=1, index_col='description'))

9300

In [15]:
rosetta_df = pd.concat(rosetta_data)
rosetta_df.head()

,SCORE:,total_score,avg_deg_4_5,avg_deg_8,avg_sc_nbrs_res_summary,bb_internal_hbonds,bbsc_internal_hbonds,cav_vol,dslf_fa13,fa_atr,...,packstat,polar_sasa,pro_close,rama_prepro,ref,sasa,sc_internal_hbonds,total_cc_contacts,yhh_planarity,secondary_structure
description,,,,,,,,,,,,,,,,,,,,,
GB_GCA_000411155.1_A_processed_0001,SCORE:,-433.888,1.351,10.876,2.716,125.0,23.0,226.760,0.0,-1110.039,...,0.578,4190.758,18.201,11.772,55.580,9931.234,27.0,631.0,0.140,LLLEEEEELLLLLLHHHHHHHHHHHHLLLEEEHHHHHHHHHHLLLH...
GB_GCA_001187505.1_A_processed_0001,SCORE:,-443.850,1.298,10.757,2.727,117.0,22.0,223.269,0.0,-1009.724,...,0.592,4151.519,11.423,-10.415,35.368,9294.481,25.0,564.0,0.076,LEEEEELLLLLLHHHHHHHHHHHHLLLEEEHHHHHHHHHHLLLHHH...
GB_GCA_001184205.1_A_processed_0001,SCORE:,-437.262,1.347,10.812,2.679,132.0,32.0,209.604,0.0,-1210.039,...,0.606,4361.120,19.617,7.042,50.130,10639.826,24.0,661.0,0.069,LEEEEELLLLLLHHHHHHHHHHHHLLLEEEHHHHHHHHHHLLLHHH...
GB_GCA_000376885.1_A_processed_0001,SCORE:,-454.076,1.278,10.765,2.648,124.0,21.0,167.472,0.0,-1068.002,...,0.576,4111.250,8.689,-13.650,39.305,9627.839,21.0,576.0,0.040,LEEEEELLLLLLHHHHHHHHHHHHLLLEEEHHHHHHHHHHLLLHHH...
GB_GCA_000423665.1_A_processed_0001,SCORE:,-474.750,1.330,10.817,2.719,141.0,30.0,338.336,0.0,-1250.547,...,0.566,5163.576,20.205,-10.989,1.885,11168.009,29.0,712.0,0.098,LEEEEEELLLLLLHHHHHHHHHHHHLLLEEEHHHHHHHHHHLLLHH...


In [16]:
# parse rosetta outputs
rosetta_df['genome_id'] = ['_'.join(s.split('_adk')[0].split('_A_pro')[0].split('_')[-3:]) for s in rosetta_df.index] #rosetta_df.index.map(lambda x: x.partition('.')[0])

# make a secondary structure series
secondary_structure = rosetta_df.groupby('genome_id').first()['secondary_structure']

# add rosetta metrics
rosetta_df['%S'] = rosetta_df['secondary_structure'].map(lambda x: x.count('E') / len(x))
rosetta_df['%L'] = rosetta_df['secondary_structure'].map(lambda x: x.count('L') / len(x))
rosetta_df['%H'] = rosetta_df['secondary_structure'].map(lambda x: x.count('H') / len(x))
rosetta_df['length'] = rosetta_df['secondary_structure'].map(len)
rosetta_df['avg_cc_contacts_per_res'] = rosetta_df['total_cc_contacts'] / rosetta_df['length']
rosetta_df = rosetta_df.groupby('genome_id').mean(numeric_only=True)

## Structure plotting code

In [17]:
import IPython

In [18]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

<IPython.core.display.Javascript object>

### Example

In [19]:
from adk_drawer import get_guiding_structures, ss_to_domains, draw_adk

ss_cmap = {'H':'red', 'E':'blue', 'L':'gray'}
domain_cmap = {'core':'red', 'lid':'blue', 'nmp':'green', 'none':'gray', 'ploop':'yellow'}
nglview_list = lambda cmap, residue_ss: [[cmap[r], str(i)] for i, r in enumerate(residue_ss)]

In [108]:
from pdb_getter import collapse_and_label

In [20]:

def AssessConf(file: str, plot: bool=False, verbose: bool=False):
    '''
    file : str -> filepath of pdb file
    plot : bool -> draw an interactive widget?
    '''
    # load trajectory
    traj = pt.load(file)
    w = nv.show_pytraj(traj)
    
    # print('distance and saving')
    distance = pt.distance(traj, ':119-154 :30-67')
    conf = 'closed'
    if distance >= 25:
        conf = 'open'
    
    if verbose:
        print('Core-lid distance: ', distance[0], f' likely conf is {conf}')
    
    # color residues in the core and lid domains
    scheme = nv.color._ColorScheme([['green', '119-154'], ['yellow', '30-67'], ['#FFA500', '7-13']], label='domains')
    
    w.clear()
    w.add_cartoon(color=scheme)
    
    # adding arrow
    p1 = list(pt.center_of_mass(traj, ':119-154')[0])
    p2 = list(pt.center_of_mass(traj, ':30-67')[0])
    w.shape.add_arrow(p1, p2, [1,0,0], 1.0)

    if plot:
        display(w)

    return distance[0]

In [25]:
def nw(x, y, match = 1, mismatch = 1, gap = 1):
    nx = len(x)
    ny = len(y)
    # Optimal score at each possible pair of characters.
    F = np.zeros((nx + 1, ny + 1))
    F[:,0] = np.linspace(0, -nx * gap, nx + 1)
    F[0,:] = np.linspace(0, -ny * gap, ny + 1)
    # Pointers to trace through an optimal aligment.
    P = np.zeros((nx + 1, ny + 1))
    P[:,0] = 3
    P[0,:] = 4
    # Temporary scores.
    t = np.zeros(3)
    for i in range(nx):
        for j in range(ny):
            if x[i] == y[j]:
                t[0] = F[i,j] + match
            else:
                t[0] = F[i,j] - mismatch
            t[1] = F[i,j+1] - gap
            t[2] = F[i+1,j] - gap
            tmax = np.max(t)
            F[i+1,j+1] = tmax
            if t[0] == tmax:
                P[i+1,j+1] += 2
            if t[1] == tmax:
                P[i+1,j+1] += 3
            if t[2] == tmax:
                P[i+1,j+1] += 4
    # Trace through an optimal alignment.
    i = nx
    j = ny
    rx = []
    ry = []
    while i > 0 or j > 0:
        if P[i,j] in [2, 5, 6, 9]:
            rx.append(x[i-1])
            ry.append(y[j-1])
            i -= 1
            j -= 1
        elif P[i,j] in [3, 5, 7, 9]:
            rx.append(x[i-1])
            ry.append('-')
            i -= 1
        elif P[i,j] in [4, 6, 7, 9]:
            rx.append('-')
            ry.append(y[j-1])
            j -= 1
    # Reverse the strings.
    rx = ''.join(rx)[::-1]
    ry = ''.join(ry)[::-1]
    return '\n'.join([rx, ry])

In [464]:

genome = 'GB_GCA_000016765.1'
file = f'{genome}_closed.pdb'
struct_path = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/test_structures/'

draw_adk(f'{struct_path}{file}', color_ss=ss_cmap, ss=secondary_structure.loc[genome])

NGLWidget()

In [460]:
# old
# 'CCCCNNNCCCLLLLLLCCC'
# 'EHEHHHEHEHEEEEEEHEH'
canonical_dom =        'CCPCCCNNNNNCCCCCCCCCLLLLLLLLLLLLLLCCCCCC'
canonical_structures = 'LELHLEHLHLHLHLELHLELHLELELELELELELHLELHL'

len(canonical_dom), len(canonical_structures)

def compute_domains(ss, m=1, mm=1, g=1):
    structure = ''.join([item[0] for key, item in ss_to_domains(ss)[0].items()])

    print(structure)
    print(len(structure))
    alignment = nw(structure, canonical_structures, match=m, mismatch=mm, gap=g)
    text = alignment.split('\n')[1]
    new_can_dom = canonical_dom
    for ind in [n for n in range(len(text)) if text.find('-', n) == n]:
        new_can_dom = new_can_dom[:ind] + new_can_dom[ind-1] + new_can_dom[ind:]
    
    print('\n'.join([alignment, canonical_dom, new_can_dom]))
    
    ss2dom = list(zip(alignment.split('\n')[0], new_can_dom))

    temp_dict = {}
    map_dict = {}
    for item in [i for i in ss2dom if not '-' in i[0]]:
        if item[0] in temp_dict.keys():
            temp_dict[item[0]] += 1
        else:
            temp_dict[item[0]] = 1
    
        map_dict[f'{item[0]}{temp_dict[item[0]]}'] = item[1]

    r2s = ss_to_domains(secondary_structure.loc[genome])[0]

    structure_loc = []
    for key, item in r2s.items():
        indeces = [int(k) for k in key.split('-')]
        structure_loc += [map_dict[item]] * len(list(range(indeces[0], indeces[1])))

    return structure_loc

In [473]:
genome = 'GB_GCA_000376885.1'
file = f'{genome}_A_processed.pdb'
struct_path = '/data/mhoffert/fiererlab/adenylate_kinase_ogt/data/processed_structures/'
pdb = f'{struct_path}{file}'

In [478]:
structure_loc = compute_domains(secondary_structure.loc[genome], mm=2)

LELHLEHLHLHLHLELHLELHLHLELHL
28
LELHLEHLHLHLHLELHLELH------------LHLELHL
LELHLEHLHLHLHLELHLELHLELELELELELELHLELHL
CCPCCCNNNNNCCCCCCCCCLLLLLLLLLLLLLLCCCCCC
CCPCCCNNNNNCCCCCCCCCLLLLLLLLLLLLLLCCCCCC


In [475]:
# load pdb
pdb = f'{struct_path}{file}'
traj = pt.load(pdb)
w = nv.show_pytraj(traj, default_representation=False)
w.add_cartoon(color='gray')




# s2r, sa = ss_to_domains(ss)
c2core = {'C':'core', 'L':'lid', 'N':'nmp', 'P':'ploop'}
sa = [c2core[i] for i in structure_loc]
colors = nglview_list(domain_cmap, sa)

scheme = nv.color._ColorScheme(colors, label='domains')

w.add_cartoon(color=scheme)

display(w)

NGLWidget()